In [ ]:
!mkdir -p benchmark-results
!cd .. && timeout 120s ./build-release/bin/benchmarks --benchmark_filter='^BM_SQL<(InterpretedExpressionExecutor|CachedJitCompiledExpressionExecutor), (kSimpleSelectSmall|kJoinSmall|kComplex5), PlannerMode::(kNaive|kOptimized)>/real_time$' --benchmark_repetitions=3 --benchmark_display_aggregates_only=true --benchmark_out=research/benchmark-results/query.json --benchmark_out_format=json

: 

In [ ]:
!cd .. && SSB_DATA_DIR=benchmarks/datasets/ssb/generated/sf001 timeout 180s ./build-release/bin/benchmarks --benchmark_filter='^SSB/' --benchmark_repetitions=3 --benchmark_display_aggregates_only=true --benchmark_out=research/benchmark-results/ssb-sf001.json --benchmark_out_format=json

In [ ]:
!cd .. && timeout 300s ./build-release/bin/benchmarks --benchmark_filter='^OperatorCost/' --benchmark_repetitions=3 --benchmark_display_aggregates_only=true --benchmark_out=research/benchmark-results/operator-cost.json --benchmark_out_format=json

In [ ]:
!cd .. && timeout 300s ./build-release/bin/benchmarks --benchmark_filter='^OperatorCostMatched/' --benchmark_repetitions=3 --benchmark_display_aggregates_only=true --benchmark_out=research/benchmark-results/operator-cost-matched.json --benchmark_out_format=json

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("benchmark-results")
UNIT_TO_MS = {"ns": 1e-6, "us": 1e-3, "ms": 1.0, "s": 1e3}

def mean_rows(filename):
    data = json.loads((RESULTS / filename).read_text())
    return [row for row in data["benchmarks"] if row.get("aggregate_name") == "mean"]

def time_ms(row):
    return row["real_time"] * UNIT_TO_MS[row["time_unit"]]

In [ ]:
synthetic_pattern = re.compile(
    r"BM_SQL<(?P<executor>[^,]+), (?P<query>[^,]+), PlannerMode::k(?P<mode>[^>]+)>/real_time_mean"
)
synthetic = []
for row in mean_rows("query.json"):
    match = synthetic_pattern.fullmatch(row["name"])
    synthetic.append({**match.groupdict(), "time_ms": time_ms(row)})
synthetic = pd.DataFrame(synthetic)

ssb = []
for row in mean_rows("ssb-sf001.json"):
    _, query, executor, mode, _ = row["name"].split("/")
    ssb.append({"query": query, "executor": executor, "mode": mode, "time_ms": time_ms(row)})
ssb = pd.DataFrame(ssb)

synthetic, ssb

In [ ]:
operator = []
for row in mean_rows("operator-cost.json"):
    parts = row["name"].split("/")
    operator.append({
        "operator": parts[1],
        "input_rows": max(map(int, parts[2:-1])),
        "model_cost": row["model_cost"],
        "time_ms": time_ms(row),
    })
operator = pd.DataFrame(operator)

matched = []
for row in mean_rows("operator-cost-matched.json"):
    parts = row["name"].split("/")
    matched.append({
        "target_cost": int(parts[1].split(":")[1]),
        "operator": parts[2],
        "input_rows": max(map(int, parts[3:-1])),
        "model_cost": row["model_cost"],
        "time_ms": time_ms(row),
    })
matched = pd.DataFrame(matched)

operator, matched

In [ ]:
ssb_interpreted = ssb[ssb.executor == "Interpreted"].copy()

wide = ssb_interpreted.pivot(index="query", columns="mode", values="time_ms").sort_index()
ax = wide[["Naive", "Optimized"]].plot.bar(figsize=(14, 5))
ax.set_ylabel("Execution time, ms")
ax.set_xlabel("Query")
ax.set_title("Naive and optimized physical plans (SSB)")
plt.tight_layout()

In [ ]:
speedup = (wide["Naive"] / wide["Optimized"]).sort_index()
ax = speedup.plot.bar(figsize=(14, 4))
ax.axhline(1.0, color="black", linewidth=1)
ax.set_ylabel("Speedup, Naive / Optimized")
ax.set_xlabel("Query")
ax.set_title("Optimizer speedup")
plt.tight_layout()

In [ ]:
jit = synthetic[synthetic["mode"] == "Optimized"].copy()
jit["executor"] = jit["executor"].replace({
    "InterpretedExpressionExecutor": "Interpreted",
    "CachedJitCompiledExpressionExecutor": "Cached JIT",
})
ax = jit.pivot(index="query", columns="executor", values="time_ms").plot.bar(figsize=(9, 4))
ax.set_ylabel("Execution time, ms")
ax.set_xlabel("Query")
ax.set_title("Expression evaluation for optimized plans")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, group in operator.groupby("operator"):
    group = group.sort_values("input_rows")
    ax.plot(group.input_rows, group.time_ms, marker="o", label=name)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Input rows")
ax.set_ylabel("Execution time, ms")
ax.set_title("Physical operator scaling")
ax.legend()
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, group in matched.groupby("operator"):
    ax.scatter(group.model_cost, group.time_ms, label=name)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Model cost")
ax.set_ylabel("Execution time, ms")
ax.set_title("Cost-model calibration")
ax.legend()
plt.tight_layout()

In [ ]:
coefficients = pd.DataFrame([
    ("SeqScan", "row", 100),
    ("Filter", "output row", 100),
    ("Projection", "output row", 22),
    ("Sort", "n * (floor(log2(n)) + 1)", 11),
    ("Aggregation", "input row", 510),
    ("NestedLoopJoin", "input row pair", 70),
    ("NestedLoopCrossJoin", "input row pair", 104),
    ("HashJoin", "build row * width", 100),
    ("HashJoin", "probe row", 35),
    ("HashJoin", "output row * width", 10),
], columns=["operator", "coefficient", "value"])
coefficients